This notebook is self-contained and should be run on https://jupyter-hub.io-ancotel.local:8443 for open-source models (GPU), or anywhere else for API-only models.

In [ ]:
from __future__ import annotations

import asyncio
import datetime
import gc
import math
import os
import re
import subprocess
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

In [ ]:
# Current time for output file naming
cet = datetime.timezone(datetime.timedelta(hours=1))
now = datetime.datetime.now(tz=cet).strftime("%Y%m%d_%H%M%S")

In [ ]:
# GPU configuration
GPU_MODE = "single"  # "single" (best free GPU), "multi" (N best), "all" (every GPU)
NUM_GPUS = None  # only used when GPU_MODE="multi"

try:
    _smi = subprocess.run(
        [
            "/usr/bin/nvidia-smi",
            "--query-gpu=memory.free",
            "--format=csv,nounits,noheader",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    _free = [int(x) for x in _smi.stdout.strip().splitlines()]

    if GPU_MODE == "single":
        _gpu = _free.index(max(_free))
        os.environ["CUDA_VISIBLE_DEVICES"] = str(_gpu)
        print(f"Using physical GPU {_gpu} ({_free[_gpu]} MiB free)")
    elif GPU_MODE == "multi":
        _ranked = sorted(range(len(_free)), key=lambda i: _free[i], reverse=True)
        _selected = _ranked[:NUM_GPUS]
        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in _selected)
        print(
            f"Using physical GPUs {_selected} (free: {[_free[g] for g in _selected]} MiB)"
        )
    elif GPU_MODE == "all":
        print(f"Using all {len(_free)} GPUs (free: {_free} MiB)")
    else:
        msg = f"Unknown GPU_MODE: {GPU_MODE!r}"
        raise ValueError(msg)
except FileNotFoundError:
    print("No NVIDIA GPU detected — API-only mode")

In [ ]:
# tiktoken configuration
repo_root = Path.cwd().parent  # adjust as needed
os.environ["TIKTOKEN_CACHE_DIR"] = str(repo_root / "tiktoken_cache")

In [ ]:
import numpy as np
import openai
import pandas as pd
import seaborn as sns
import tiktoken
import torch
import transformers
from datasets import load_dataset
from dotenv import load_dotenv
from matplotlib import pyplot as plt
from openai import AsyncOpenAI
from peft import PeftModel
from scipy.stats import ttest_1samp, wilcoxon
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# Filter which models to run: "all", "api", or "hf"
RUN_MODE = "api"

CONFIG = {
    "paths": {
        "data": Path("../data/processed"),
        "output": Path("../data/output"),
        "ft_adapter": Path("../models/Qwen2.5-7B-Instruct-FT"),
    },
    "model": {"attn_implementation": "flash_attention_2"},
    "stability": {
        "n_trials": 20,
        "hf_trial_batch_size": 16,
        "temperatures": [0.0, 0.5, 1.0, 2.0],
        "n_candidates": "c250",
        "max_new_tokens": 256,
        "k_values": [1, 5, 10],
        "top_p": 1.0,
        "sampled_prompt_indices": [
            640,
            515,
            421,
            837,
            86,
            92,
            181,
            820,
            774,
            681,
            126,
            197,
            957,
            506,
            829,
            776,
            367,
            722,
            400,
            427,
            442,
            544,
            445,
            495,
            751,
            920,
            705,
            636,
            83,
            748,
        ],
    },
    "api": {"max_concurrent": 3, "encoding": "o200k_base"},
    "seed": 42,
}

## Model registry

In [ ]:
@dataclass
class ModelConfig:
    """Configuration for a single model."""

    name: str
    model_type: str  # "local_hf" | "local_hf_ft" | "api"
    model_id: str
    candidate_counts: list[str] = field(default_factory=list)
    context_window: int = 128_000
    quantization: str | None = None
    adapter_path: str | None = None
    dtype: str = "bfloat16"
    supports_system_role: bool = True
    supports_sampling: bool = True
    extra_api_params: dict = field(default_factory=dict)
    max_temperature: float = 2.0  # Claude models get 1.0


MODEL_REGISTRY: list[ModelConfig] = [
    # Open-source (GPU server)
    ModelConfig(
        name="Llama-3.2-3B",
        model_type="local_hf",
        model_id="meta-llama/Llama-3.2-3B-Instruct",
        candidate_counts=["c250"],
    ),
    ModelConfig(
        name="Llama-3.1-8B",
        model_type="local_hf",
        model_id="meta-llama/Llama-3.1-8B-Instruct",
        candidate_counts=["c250"],
    ),
    ModelConfig(
        name="Llama-3.3-70B",
        model_type="local_hf",
        model_id="meta-llama/Llama-3.3-70B-Instruct",
        candidate_counts=["c250"],
        quantization="4bit_bnb",
    ),
    ModelConfig(
        name="Qwen2.5-7B",
        model_type="local_hf",
        model_id="Qwen/Qwen2.5-7B-Instruct",
        candidate_counts=["c250"],
    ),
    ModelConfig(
        name="Qwen2.5-7B-FT",
        model_type="local_hf_ft",
        model_id="Qwen/Qwen2.5-7B-Instruct",
        candidate_counts=["c250"],
        adapter_path=CONFIG["paths"]["ft_adapter"],
    ),
    ModelConfig(
        name="Gemma-2-9B",
        model_type="local_hf",
        model_id="google/gemma-2-9b-it",
        candidate_counts=["c250"],
        supports_system_role=False,
    ),
    # API models (via LiteLLM)
    ModelConfig(
        name="infobip-gpt-4-1",
        model_type="api",
        model_id="infobip-gpt-4-1",
        candidate_counts=["c250"],
    ),
    ModelConfig(
        name="infobip-gpt-4-1-mini",
        model_type="api",
        model_id="infobip-gpt-4-1-mini",
        candidate_counts=["c250"],
    ),
    ModelConfig(
        name="gpt-5.2",
        model_type="api",
        model_id="gpt-5.2",
        candidate_counts=["c250"],
        extra_api_params={"reasoning_effort": "none"},
    ),
    ModelConfig(
        name="claude-sonnet-4-6",
        model_type="api",
        model_id="claude-sonnet-4-6",
        candidate_counts=["c250"],
        context_window=200_000,
        max_temperature=1.0,
    ),
    ModelConfig(
        name="claude-opus-4-6",
        model_type="api",
        model_id="claude-opus-4-6",
        candidate_counts=["c250"],
        context_window=200_000,
        max_temperature=1.0,
    ),
]

## Seed, environment, API client

In [ ]:
# Seed
SEED = CONFIG["seed"]
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

# Enable TF32 for faster matmul on Ampere+ GPUs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.fp32_precision = "tf32"
    torch.backends.cudnn.conv.fp32_precision = "tf32"

# Environment
load_dotenv()

# API client
base_url = os.getenv("LITELLM_ENDPOINT", "").rstrip("/")
if base_url and not base_url.endswith("/v1"):
    base_url += "/v1"
api_client = AsyncOpenAI(
    base_url=base_url,
    api_key=os.getenv("LITELLM_API_KEY"),
)
print(f"API client base_url: {base_url}")

# Tiktoken encoder for entropy
enc = tiktoken.get_encoding(CONFIG["api"]["encoding"])

## Utility functions

> **Note:** Functions below are copied from `src/stability/` to keep this notebook
> self-contained for remote JupyterHub execution.

In [ ]:
# Copied from src/stability/utils.py
def canonicalize(s: str | None) -> str | None:
    """Return a canonicalized version of the string for comparison."""
    if s is None:
        return None
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\u2014", "-").replace("\u2013", "-")
    s = "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )
    result = s.strip().strip('"').strip("'").rstrip(".,:;!?").lower()
    result = re.sub(r"\s+", " ", result)
    return result.replace("&", "and")

In [ ]:
# Copied from src/stability/generation.py
def extract_pred_items(response: str, max_items: int = 10) -> list[str]:
    """Extract predicted items (movie titles) from raw LLM response text."""
    response = response.strip()
    lines = [
        re.sub(r"^\d+[.)]\s*", "", ln).strip(" -*\u2022\t")
        for ln in response.splitlines()
        if ln.strip()
    ]

    items: list[str] = []
    for ln in lines:
        if ln and 2 <= len(ln) <= 120:
            items.append(ln)
        if len(items) >= max_items:
            break

    seen: set[str | None] = set()
    dedup: list[str] = []
    for it in items:
        it_canon = canonicalize(it)
        if it_canon and it_canon not in seen:
            seen.add(it_canon)
            dedup.append(it)
    return dedup[:max_items]

In [ ]:
# Copied from src/stability/metrics.py (standalone version)
def recommendation_metrics(
    predictions: list[str],
    ground_truth: list[str],
    k_values: list[int],
) -> dict[str, float]:
    """Return hit_rate@K, mrr@K, precision@K, recall@K, f1@K, ndcg@K."""
    metrics: dict[str, float] = {}
    if not ground_truth:
        for k in k_values:
            for name in ["hit_rate", "mrr", "precision", "recall", "f1", "ndcg"]:
                metrics[f"{name}@{k}"] = np.nan
        return metrics

    truth_canon = [canonicalize(t) for t in ground_truth if t]
    pred_canon = [canonicalize(p) for p in predictions if p]

    for k in k_values:
        topk = pred_canon[:k]
        relevance = np.array([1 if p in truth_canon else 0 for p in topk])

        hit_rate_k = 1.0 if relevance.sum() > 0 else 0.0
        mrr_k = 0.0
        if relevance.any():
            mrr_k = 1.0 / (np.argmax(relevance) + 1)

        tp = relevance.sum()
        precision_k = tp / k
        recall_k = tp / len(truth_canon)
        f1_k = (
            2 * precision_k * recall_k / (precision_k + recall_k)
            if (precision_k + recall_k) > 0
            else 0.0
        )

        dcg = np.sum(relevance / np.log2(np.arange(2, len(relevance) + 2)))
        n_relevant = min(len(truth_canon), k)
        ideal_relevance = np.zeros(k)
        ideal_relevance[:n_relevant] = 1
        idcg = np.sum(ideal_relevance / np.log2(np.arange(2, k + 2)))
        ndcg_k = dcg / idcg if idcg > 0 else 0.0

        metrics[f"hit_rate@{k}"] = hit_rate_k
        metrics[f"mrr@{k}"] = mrr_k
        metrics[f"precision@{k}"] = precision_k
        metrics[f"recall@{k}"] = recall_k
        metrics[f"f1@{k}"] = f1_k
        metrics[f"ndcg@{k}"] = ndcg_k
    return metrics

In [ ]:
def compute_text_entropy(text: str, encoder: tiktoken.Encoding) -> dict[str, float]:
    """Compute token-level entropy metrics from text using a tiktoken encoder."""
    token_ids = encoder.encode(text)
    if not token_ids:
        return {"entropy": 0.0, "normalized_entropy": 0.0, "unique_token_ratio": 0.0}
    unique, counts = np.unique(token_ids, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs)))
    max_entropy = math.log2(len(unique)) if len(unique) > 1 else 1.0
    return {
        "entropy": entropy,
        "normalized_entropy": entropy / max_entropy if max_entropy > 0 else 0.0,
        "unique_token_ratio": len(unique) / len(token_ids),
    }

In [ ]:
# Copied from src/stability/utils.py
def bootstrap_mean_ci(
    values: np.ndarray,
    rng: np.random.Generator,
    n_boot: int = 1000,
    ci: float = 95.0,
) -> tuple[float, float, float]:
    """Return mean and bootstrap confidence interval for the given values."""
    vals = values[np.isfinite(values)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    mean_val = float(np.mean(vals))
    if len(vals) == 1:
        return mean_val, mean_val, mean_val

    boots = [
        rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)
    ]
    low = np.percentile(boots, (100 - ci) / 2)
    high = np.percentile(boots, 100 - (100 - ci) / 2)
    return mean_val, float(low), float(high)

In [ ]:
# Copied from src/stability/metrics.py — diversity metrics
def cosine_distance_matrix(embeds: np.ndarray) -> np.ndarray:
    """Return cosine distance matrix D = 1 - cosine_similarity, with zero diagonal."""
    normalized = embeds / np.linalg.norm(embeds, axis=1, keepdims=True)
    similarity = normalized @ normalized.T
    distance = 1.0 - similarity
    np.fill_diagonal(distance, 0.0)
    return distance


def label_metrics(counts: np.ndarray) -> dict[str, float]:
    """Return label distribution metrics: gini, entropy, variation_ratio, unique_count."""
    n_total = counts.sum()
    if n_total <= 0:
        return {
            "gini": np.nan,
            "entropy": np.nan,
            "variation_ratio": np.nan,
            "unique_count": 0.0,
        }
    probs = counts / n_total
    sorted_probs = np.sort(probs)
    n_items = len(probs)

    gini = (
        2 * np.sum(np.arange(1, n_items + 1) * sorted_probs) / np.sum(probs)
        - n_items
        - 1
    ) / n_items

    probs_nonzero = probs[probs > 0]
    probs_clipped = np.clip(probs_nonzero, 1e-10, 1.0)
    entropy = float(-np.sum(probs_nonzero * np.log(probs_clipped)))

    variation_ratio = 1.0 - float(np.max(probs))
    unique_count = float(np.count_nonzero(counts))

    return {
        "gini": gini,
        "entropy": entropy,
        "variation_ratio": variation_ratio,
        "unique_count": unique_count,
    }


def expected_distance_from_counts(
    counts: np.ndarray,
    distance_matrix: np.ndarray,
) -> tuple[float, float]:
    """Return expected embedding distance E = p^T D p and its normalization."""
    n_total = counts.sum()
    if n_total <= 0:
        return np.nan, np.nan
    probs = counts / n_total
    expected_dist = float(probs @ distance_matrix @ probs)
    n_items = len(probs)
    if n_items <= 1:
        return expected_dist, np.nan
    uniform_probs = np.ones(n_items) / n_items
    uniform_dist = float(uniform_probs @ distance_matrix @ uniform_probs)
    normalized_dist = expected_dist / uniform_dist if uniform_dist > 0 else np.nan
    return expected_dist, normalized_dist


def mean_pairwise_cosine_distance(embeds: np.ndarray) -> float:
    """Return mean pairwise cosine distance over embeddings."""
    n_samples = embeds.shape[0]
    if n_samples <= 1:
        return np.nan
    normalized = embeds / np.linalg.norm(embeds, axis=1, keepdims=True)
    similarity = normalized @ normalized.T
    upper_indices = np.triu_indices(n_samples, k=1)
    mean_sim = float(similarity[upper_indices].mean())
    return 1.0 - mean_sim


def cosine_diversity(embeds: np.ndarray) -> float:
    """Calculate cosine diversity as 1 - mean cosine similarity to centroid."""
    n_samples = embeds.shape[0]
    if n_samples <= 1:
        return np.nan
    normalized = embeds / np.linalg.norm(embeds, axis=1, keepdims=True)
    centroid = np.mean(normalized, axis=0)
    centroid_normalized = centroid / np.linalg.norm(centroid)
    similarities = np.dot(normalized, centroid_normalized)
    return 1.0 - np.mean(similarities)

## Prompt builders

In [ ]:
def merge_system_into_user(messages: list[dict[str, str]]) -> list[dict[str, str]]:
    """Merge system message content into the first user message."""
    system_parts = [m["content"] for m in messages if m["role"] == "system"]
    other = [m for m in messages if m["role"] != "system"]
    if not system_parts:
        return other
    prefix = "\n\n".join(system_parts)
    return [
        {**msg, "content": prefix + "\n\n" + msg["content"]}
        if msg["role"] == "user"
        else msg
        for msg in other
    ]


def build_hf_prompt(
    messages: list[dict[str, str]],
    tokenizer: Any,  # noqa: ANN401
    sep: str,
    supports_system_role: bool = True,
) -> str:
    """Apply chat template and strip at the assistant separator."""
    if not supports_system_role:
        messages = merge_system_into_user(messages)
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    if sep in formatted:
        idx = formatted.index(sep) + len(sep)
        return formatted[:idx]
    return formatted

In [ ]:
def build_api_messages(messages: list[dict[str, str]]) -> list[dict[str, str]]:
    """Strip the assistant message, return system+user for chat completions."""
    return [m for m in messages if m["role"] != "assistant"]


def detect_assistant_separator(
    tokenizer: Any,  # noqa: ANN401
    supports_system_role: bool = True,
) -> str:
    """Return the separator string for the assistant role from the chat template."""
    test_messages = [
        {"role": "user", "content": "test"},
        {"role": "assistant", "content": "test"},
    ]
    if supports_system_role:
        test_messages.insert(0, {"role": "system", "content": "test"})
    formatted = tokenizer.apply_chat_template(test_messages, tokenize=False)
    patterns = [
        ("llama3", "<|start_header_id|>assistant<|end_header_id|>\n\n"),
        ("qwen", "<|im_start|>assistant\n"),
        ("phi3", "<|assistant|>\n"),
        ("gemma", "model\n"),
        ("mistral", "[/INST]"),
    ]
    for name, pattern in patterns:
        if pattern in formatted:
            print(f"Detected {name} chat template, separator: {pattern!r}")
            return pattern

    if "<|start_header_id|>assistant<|end_header_id|>" in formatted:
        return "<|start_header_id|>assistant<|end_header_id|>\n\n"
    if "<|im_start|>assistant" in formatted:
        return "<|im_start|>assistant\n"
    if "<|assistant|>" in formatted:
        return "<|assistant|>\n"
    if "[/INST]" in formatted:
        return "[/INST]"

    msg = (
        "Could not automatically detect the assistant separator.\n"
        f"Formatted test messages:\n{formatted}\n"
        "Please inspect the above output to determine the correct separator."
    )
    raise ValueError(msg)

## Generators

In [ ]:
@torch.no_grad()
def generate_hf(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompts: list[str],
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_input_length: int = 8192,
) -> list[dict[str, str | float]]:
    """Batch HF generation with entropy computation."""
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_input_length,
    ).to(model.device)

    if inputs["input_ids"].shape[1] >= max_input_length:
        print(f"  WARNING: Input truncated to {max_input_length} tokens")

    gen_kwargs = {
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "max_new_tokens": max_new_tokens,
        "output_scores": True,
        "return_dict_in_generate": True,
    }
    if temperature > 0:
        gen_kwargs.update(
            {"do_sample": True, "temperature": temperature, "top_p": top_p}
        )
    else:
        gen_kwargs["do_sample"] = False

    outputs = model.generate(**inputs, **gen_kwargs)
    input_len = inputs["input_ids"].shape[1]

    results: list[dict[str, str | float]] = []
    for i, output_ids in enumerate(outputs.sequences):
        response_ids = output_ids[input_len:]
        response = tokenizer.decode(response_ids, skip_special_tokens=True).strip()

        logits = torch.stack(outputs.scores, dim=1)[i, :, :]
        probs = torch.nn.functional.softmax(logits, dim=-1)
        token_entropy = -torch.sum(probs * torch.clamp(probs, min=1e-10).log2(), dim=-1)
        avg_entropy = token_entropy.mean().item()
        norm_entropy = avg_entropy / math.log2(probs.shape[-1])

        results.append(
            {
                "response": response,
                "entropy": avg_entropy,
                "normalized_entropy": norm_entropy,
                "unique_token_ratio": len(set(response_ids.tolist()))
                / max(len(response_ids), 1),
            }
        )
    return results

In [ ]:
async def generate_api_trials(
    client: AsyncOpenAI,
    model: str,
    messages: list[dict[str, str]],
    n_trials: int,
    seed_base: int,
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_concurrent: int = 10,
    extra_body: dict | None = None,
) -> list[dict[str, str | float]]:
    """Generate n_trials responses for the same prompt with varying seeds.

    Each trial gets seed = seed_base + trial_idx. Returns list of dicts
    with 'response', 'entropy', 'normalized_entropy', 'unique_token_ratio'.
    """
    semaphore = asyncio.Semaphore(max_concurrent)
    completed = 0

    async def call_api(trial_idx: int) -> tuple[int, dict]:
        nonlocal completed
        async with semaphore:
            text = ""
            entropy_metrics = {
                "entropy": 0.0,
                "normalized_entropy": 0.0,
                "unique_token_ratio": 0.0,
            }
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    create_kwargs: dict[str, Any] = {
                        "model": model,
                        "messages": messages,
                        "temperature": temperature,
                        "max_tokens": max_new_tokens,
                        "seed": seed_base + trial_idx,
                        **(extra_body or {}),
                    }
                    if top_p != 1.0:
                        create_kwargs["top_p"] = top_p
                    response = await client.chat.completions.create(**create_kwargs)
                    text = (response.choices[0].message.content or "").strip()
                    entropy_metrics = compute_text_entropy(text, enc)
                    break
                except (
                    openai.APIError,
                    openai.APIConnectionError,
                    openai.RateLimitError,
                    openai.APITimeoutError,
                ) as e:
                    wait = 2**attempt
                    if attempt < max_retries - 1:
                        print(
                            f"API error trial {trial_idx} (attempt {attempt + 1}): {e}, retrying in {wait}s"
                        )
                        await asyncio.sleep(wait)
                    else:
                        print(
                            f"API error trial {trial_idx} (attempt {attempt + 1}): {e}, giving up"
                        )
            completed += 1
            return trial_idx, {"response": text, **entropy_metrics}

    tasks = [call_api(i) for i in range(n_trials)]
    results = await asyncio.gather(*tasks)
    results_sorted = sorted(results, key=lambda x: x[0])
    return [r for _, r in results_sorted]

## Model loading / unloading

In [ ]:
def load_hf_model(cfg: ModelConfig) -> tuple[AutoModelForCausalLM, Any]:
    """Load a HuggingFace model (optionally with 4-bit quant and/or LoRA)."""
    dtype = getattr(torch, cfg.dtype)
    load_kwargs = {
        "dtype": dtype,
        "device_map": "auto",
        "trust_remote_code": True,
        "attn_implementation": CONFIG["model"]["attn_implementation"],
    }

    if cfg.quantization == "4bit_bnb":
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
        )

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(cfg.model_id, **load_kwargs)

    if cfg.model_type == "local_hf_ft" and cfg.adapter_path:
        model = PeftModel.from_pretrained(model, cfg.adapter_path)
        model = model.merge_and_unload()
        print(f"  LoRA adapter merged from {cfg.adapter_path}")

    model.eval()

    for c in (model.config, model.generation_config):
        c.pad_token_id = tokenizer.pad_token_id
        c.eos_token_id = tokenizer.eos_token_id

    model.generation_config.temperature = 1.0
    model.generation_config.top_p = 1.0

    print(f"  Loaded {cfg.name} ({cfg.model_id}, quant={cfg.quantization})")
    return model, tokenizer


def unload_model(model: Any, tokenizer: Any) -> None:  # noqa: ANN401
    """Free GPU memory."""
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("  Model unloaded, CUDA cache cleared")

## Data loading

In [ ]:
# Load c250 dataset (999 examples)
DATA_PATH = CONFIG["paths"]["data"]
n_candidates = CONFIG["stability"]["n_candidates"]
dataset_path = DATA_PATH / f"test_prompt_examples_{n_candidates}_r10.jsonl"
dataset = load_dataset("json", data_files=dataset_path.as_posix())["train"]
n_examples = len(dataset)
print(f"Loaded {n_examples} examples from {dataset_path.name}")

In [ ]:
# Load movies CSV + embeddings
movies_df = pd.read_csv(DATA_PATH / "movies_with_mentions_processed.csv")
all_movie_titles = movies_df["title_norm"].unique().tolist()
print(f"Loaded {len(all_movie_titles)} unique movie titles")

embeddings = np.load(DATA_PATH / "movies_embeds_extended_all-mpnet-base-v2.npy")
print(f"Loaded embeddings: {embeddings.shape}")

# Build options mapping: canonicalized -> title_norm
options_mapping: dict[str, str] = {}
for title in all_movie_titles:
    c = canonicalize(title)
    if c is not None:
        options_mapping[c] = title

# Align movies with embeddings (handle 8-row mismatch)
# Use only titles present in both CSV and embeddings
n_embed = embeddings.shape[0]
n_titles = len(all_movie_titles)
if n_embed != n_titles:
    print(
        f"  Mismatch: {n_titles} titles vs {n_embed} embeddings, using min({n_embed}, {n_titles})"
    )
    n_common = min(n_embed, n_titles)
    all_movie_titles = all_movie_titles[:n_common]
    embeddings = embeddings[:n_common]

# Build title -> embedding index mapping
title_to_idx: dict[str, int] = {t: i for i, t in enumerate(all_movie_titles)}
print(f"Aligned: {len(all_movie_titles)} titles with {embeddings.shape[0]} embeddings")

In [ ]:
# Pre-compute pairwise distance matrix and uniform baseline
d_matrix = cosine_distance_matrix(embeddings)
print(f"Distance matrix shape: {d_matrix.shape}")

n_options = len(all_movie_titles)
p_uniform = np.ones(n_options) / n_options
e_uniform = float(p_uniform @ d_matrix @ p_uniform) if n_options > 1 else 0.0
print(f"Uniform baseline expected distance: {e_uniform:.6f}")

## Prompt sampling

In [ ]:
# Fixed prompt sample used for both API and HF stability runs.
# Sampled once with np.random.default_rng(SEED).choice(999, size=30, replace=False).
SAMPLED_PROMPT_INDICES = CONFIG["stability"]["sampled_prompt_indices"]
N_PROMPTS = len(SAMPLED_PROMPT_INDICES)

if len(SAMPLED_PROMPT_INDICES) != len(set(SAMPLED_PROMPT_INDICES)):
    msg = "SAMPLED_PROMPT_INDICES must contain unique prompt indices"
    raise ValueError(msg)
if min(SAMPLED_PROMPT_INDICES) < 0 or max(SAMPLED_PROMPT_INDICES) >= n_examples:
    msg = f"SAMPLED_PROMPT_INDICES must be within dataset range 0..{n_examples - 1}"
    raise ValueError(msg)

print(f"Fixed stability prompts: {N_PROMPTS}")
print(f"Prompt indices: {SAMPLED_PROMPT_INDICES}")

## Per-trial result processing

In [ ]:
def process_single_trial(
    gen_result: dict[str, str | float],
    ground_truth: list[str],
    k_values: list[int],
) -> dict[str, Any]:
    """Extract predictions from one trial and compute quality metrics.

    Returns dict with: response, pred_items, num_pred_items,
    per-K quality metrics, entropy metrics.
    """
    response = gen_result["response"]
    pred_items = extract_pred_items(response, max_items=10)
    metrics = recommendation_metrics(pred_items, ground_truth, k_values)

    return {
        "response": response,
        "pred_items": pred_items,
        "num_pred_items": len(pred_items),
        "ground_truth": ground_truth,
        "num_ground_truth": len(ground_truth),
        "entropy": gen_result.get("entropy", 0.0),
        "normalized_entropy": gen_result.get("normalized_entropy", 0.0),
        "unique_token_ratio": gen_result.get("unique_token_ratio", 0.0),
        **metrics,
    }

## Sanity check / test run

Test with 1 API model, 2 prompts, 5 trials, 2 temperatures before the full experiment.

In [ ]:
# Sanity check: small-scale test
TEST_MODEL = "infobip-gpt-4-1"
TEST_N_PROMPTS = 2
TEST_N_TRIALS = 5
TEST_TEMPS = [0.0, 1.0]

test_cfg = next(m for m in MODEL_REGISTRY if m.name == TEST_MODEL)
test_indices = SAMPLED_PROMPT_INDICES[:TEST_N_PROMPTS]
k_values = CONFIG["stability"]["k_values"]
max_new_tokens = CONFIG["stability"]["max_new_tokens"]
top_p = CONFIG["stability"]["top_p"]

test_records = []
for temp in TEST_TEMPS:
    for pidx in test_indices:
        example = dataset[pidx]
        messages = build_api_messages(example["messages"])
        gt = example["ground_truth"]

        seed_base = SEED + pidx * 1000 + int(temp * 10) * 100
        trial_results = await generate_api_trials(
            client=api_client,
            model=test_cfg.model_id,
            messages=messages,
            n_trials=TEST_N_TRIALS,
            seed_base=seed_base,
            max_new_tokens=max_new_tokens,
            temperature=temp,
            top_p=top_p,
            max_concurrent=CONFIG["api"]["max_concurrent"],
            extra_body=test_cfg.extra_api_params or None,
        )

        for trial_idx, gen_result in enumerate(trial_results):
            row = process_single_trial(gen_result, gt, k_values)
            row |= {
                "round_idx": 0,
                "prompt_idx": pidx,
                "model": test_cfg.name,
                "model_type": test_cfg.model_type,
                "model_id": test_cfg.model_id,
                "temperature": temp,
                "trial_idx": trial_idx,
            }
            test_records.append(row)

df_test = pd.DataFrame(test_records)
print(f"Test output shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")
print(
    f"\nExpected rows: {TEST_N_PROMPTS * TEST_N_TRIALS * len(TEST_TEMPS)} = "
    f"{TEST_N_PROMPTS} prompts x {TEST_N_TRIALS} trials x {len(TEST_TEMPS)} temps"
)

# Show example
ex = df_test.iloc[0]
print("\n--- Example trial ---")
print(f"Model: {ex['model']}, temp={ex['temperature']}, trial={ex['trial_idx']}")
print(f"Response: {ex['response'][:200]}...")
print(f"Extracted items: {ex['pred_items']}")
print(f"Ground truth: {ex['ground_truth']}")
print(f"ndcg@10={ex['ndcg@10']:.4f}, hit_rate@10={ex['hit_rate@10']:.4f}")
print(f"entropy={ex['entropy']:.4f}, normalized_entropy={ex['normalized_entropy']:.4f}")

# Quick stats
print("\n--- Stats per (temp) ---")
print(df_test.groupby("temperature")[["ndcg@10", "hit_rate@10", "entropy"]].describe())

In [ ]:
print("empty", (df_test["response"].fillna("").str.len() == 0).sum())
print("num_pred_items distribution:")
print(df_test.groupby(["temperature", "num_pred_items"]).size())

print("\nUnique responses per prompt/temp:")
print(df_test.groupby(["prompt_idx", "temperature"])["response"].nunique())

print("\nDuplicate keys:")
print(df_test.duplicated(["model", "prompt_idx", "temperature", "trial_idx"]).sum())

## Main experiment loop

In [ ]:
OUTPUT_PATH = CONFIG["paths"]["output"]
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUTPUT_PATH / f"stability_{now}.log"


def log(msg: str, end: str = "\n") -> None:
    """Print to stdout and append to log file."""
    print(msg, end=end)
    if end == "\n":
        with LOG_FILE.open("a") as f:
            f.write(msg + "\n")


# Experiment parameters
N_TRIALS = CONFIG["stability"]["n_trials"]
HF_TRIAL_BATCH_SIZE = CONFIG["stability"]["hf_trial_batch_size"]
CONFIG_TEMPS = CONFIG["stability"]["temperatures"]
k_values = CONFIG["stability"]["k_values"]
max_new_tokens = CONFIG["stability"]["max_new_tokens"]
top_p = CONFIG["stability"]["top_p"]

# Resume from current-run checkpoint only, matching scripts/04_evaluation.ipynb behavior.
# Do not scan for latest partial globally: API and HF runs may execute in parallel.
checkpoint_path = OUTPUT_PATH / f"stability_results_partial_{now}.parquet"
all_records: list[dict] = []
completed_models: set[str] = set()

if checkpoint_path.exists():
    df_ckpt = pd.read_parquet(checkpoint_path)
    all_records = df_ckpt.to_dict("records")
    completed_models = set(df_ckpt["model"].unique())
    log(
        f"Resumed from checkpoint: {len(all_records)} records, "
        f"{len(completed_models)} models done"
    )

# Filter models by RUN_MODE
models_to_run = MODEL_REGISTRY
if RUN_MODE == "api":
    models_to_run = [m for m in MODEL_REGISTRY if m.model_type == "api"]
elif RUN_MODE == "hf":
    models_to_run = [m for m in MODEL_REGISTRY if m.model_type != "api"]

log(f"Models to run: {[m.name for m in models_to_run]}")
log(f"Temperatures: {CONFIG_TEMPS}")
log(f"Prompts: {N_PROMPTS}, trials per prompt-temp: {N_TRIALS}")
log(f"Prompt indices: {SAMPLED_PROMPT_INDICES}")

for cfg in models_to_run:
    if cfg.name in completed_models:
        log(f"  [{cfg.name}] already completed, skipping")
        continue

    log(f"\n{'=' * 60}")
    log(f"Model: {cfg.name} ({cfg.model_type})")
    temps = [t for t in CONFIG_TEMPS if t <= cfg.max_temperature]
    log(f"Temperatures: {temps} (max={cfg.max_temperature})")
    log(f"{'=' * 60}")

    # Load HF model if needed
    hf_model = hf_tokenizer = sep = None
    if cfg.model_type in ("local_hf", "local_hf_ft"):
        hf_model, hf_tokenizer = load_hf_model(cfg)
        sep = detect_assistant_separator(hf_tokenizer, cfg.supports_system_role)

    for temp_idx, temp in enumerate(temps):
        for p_num, prompt_idx in enumerate(SAMPLED_PROMPT_INDICES):
            example = dataset[prompt_idx]
            gt = example["ground_truth"]

            progress = f"  [{cfg.name}] temp {temp} | prompt {p_num + 1}/{N_PROMPTS}"
            print(progress, end="\r")

            # Generate n_trials responses
            is_local = cfg.model_type in ("local_hf", "local_hf_ft")

            if is_local:
                prompt_text = build_hf_prompt(
                    example["messages"],
                    hf_tokenizer,
                    sep,
                    cfg.supports_system_role,
                )
                trial_results = []
                for sub_start in range(0, N_TRIALS, HF_TRIAL_BATCH_SIZE):
                    seed = SEED + prompt_idx * 1000 + temp_idx * 100 + sub_start
                    torch.manual_seed(seed)
                    torch.cuda.manual_seed_all(seed)
                    sub_n = min(HF_TRIAL_BATCH_SIZE, N_TRIALS - sub_start)
                    results = generate_hf(
                        hf_model,
                        hf_tokenizer,
                        [prompt_text] * sub_n,
                        max_new_tokens=max_new_tokens,
                        temperature=temp,
                        top_p=top_p,
                    )
                    trial_results.extend(results)
            else:
                messages = build_api_messages(example["messages"])
                seed_base = SEED + prompt_idx * 1000 + temp_idx * 100
                trial_results = await generate_api_trials(
                    client=api_client,
                    model=cfg.model_id,
                    messages=messages,
                    n_trials=N_TRIALS,
                    seed_base=seed_base,
                    max_new_tokens=max_new_tokens,
                    temperature=temp,
                    top_p=top_p,
                    max_concurrent=CONFIG["api"]["max_concurrent"],
                    extra_body=cfg.extra_api_params or None,
                )

            # Process each trial -> one row
            for trial_idx, gen_result in enumerate(trial_results):
                row = process_single_trial(gen_result, gt, k_values)
                row |= {
                    "round_idx": 0,  # retained for compatibility; only one fixed prompt set
                    "prompt_idx": prompt_idx,
                    "model": cfg.name,
                    "model_type": cfg.model_type,
                    "model_id": cfg.model_id,
                    "temperature": temp,
                    "trial_idx": trial_idx,
                }
                all_records.append(row)

    # Checkpoint after each model
    pd.DataFrame(all_records).to_parquet(checkpoint_path, index=False)
    log(f"  [{cfg.name}] done, {len(all_records)} records saved")

    # Unload HF model
    if cfg.model_type in ("local_hf", "local_hf_ft") and hf_model is not None:
        unload_model(hf_model, hf_tokenizer)

log(f"\nExperiment complete. Total records: {len(all_records)}")

## Save results

In [ ]:
df = pd.DataFrame.from_records(all_records)
output_path = OUTPUT_PATH / f"stability_results_{now}.parquet"
df.to_parquet(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"Shape: {df.shape}")
print(f"Models: {df['model'].unique().tolist()}")
print("\nRows per model:")
print(df.groupby("model").size().to_string())
df.head()

## Analysis

Aggregate per-trial data and compute diversity metrics, summary statistics,
determinism checks, and stability rankings.

In [ ]:
# Aggregate per-trial -> per-group
# Group by (round_idx, prompt_idx, model, temperature)
# round_idx is always 0 and retained for compatibility.
# Compute diversity metrics from first-recommendations across repeated trials.
# Also compute statistical moments for quality metrics

metric_names = ["hit_rate", "mrr", "precision", "recall", "f1", "ndcg"]
quality_cols = [f"{m}@{k}" for m in metric_names for k in k_values]

agg_rows = []
groups = df.groupby(["round_idx", "prompt_idx", "model", "temperature"])

for (round_idx, prompt_idx, model_name, temp), group in groups:
    row: dict[str, Any] = {
        "round_idx": round_idx,
        "prompt_idx": prompt_idx,
        "model": model_name,
        "temperature": temp,
        "n_trials": len(group),
    }

    # Quality metric moments (mean, std, cv)
    for col in quality_cols:
        vals = group[col].dropna().to_numpy()
        if len(vals) > 0:
            mean_val = float(np.mean(vals))
            std_val = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            cv_val = std_val / mean_val if mean_val > 0 else np.nan
        else:
            mean_val = std_val = cv_val = np.nan
        row[f"{col}_mean"] = mean_val
        row[f"{col}_std"] = std_val
        row[f"{col}_cv"] = cv_val

    # Entropy moments
    for ecol in ["entropy", "normalized_entropy", "unique_token_ratio"]:
        vals = group[ecol].dropna().to_numpy()
        row[f"{ecol}_mean"] = float(np.mean(vals)) if len(vals) > 0 else np.nan
        row[f"{ecol}_std"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

    # Diversity metrics from first-recommendations
    first_recs = []
    for items in group["pred_items"]:
        if isinstance(items, list) and len(items) > 0:
            first_recs.append(items[0])
        elif isinstance(items, str):
            # Handle case where pred_items was serialized as string
            import ast

            try:
                parsed = ast.literal_eval(items)
                if parsed:
                    first_recs.append(parsed[0])
            except (ValueError, SyntaxError):
                pass

    # Count-based diversity
    counts = np.zeros(n_options, dtype=int)
    per_sample_idx: list[int] = []
    for rec in first_recs:
        c = canonicalize(rec)
        mapped = options_mapping.get(c) if c else None
        if mapped:
            idx = title_to_idx.get(mapped)
            if idx is not None:
                counts[idx] += 1
                per_sample_idx.append(idx)

    if counts.sum() > 0:
        lm = label_metrics(counts.astype(float))
        e, e_norm = expected_distance_from_counts(counts.astype(float), d_matrix)
    else:
        lm = {
            "gini": np.nan,
            "entropy": np.nan,
            "variation_ratio": np.nan,
            "unique_count": 0,
        }
        e, e_norm = np.nan, np.nan

    if len(per_sample_idx) >= 2:
        sample_embeds = embeddings[np.array(per_sample_idx)]
        mpair = mean_pairwise_cosine_distance(sample_embeds)
        cdiv = cosine_diversity(sample_embeds)
    else:
        mpair = np.nan
        cdiv = np.nan

    row["gini"] = lm["gini"]
    row["label_entropy"] = lm["entropy"]
    row["variation_ratio"] = lm["variation_ratio"]
    row["unique_count"] = lm["unique_count"]
    row["expected_distance"] = e
    row["normalized_expected_distance"] = e_norm
    row["mean_pairwise_cosine_distance"] = mpair
    row["cosine_diversity"] = cdiv

    agg_rows.append(row)

df_agg = pd.DataFrame(agg_rows)
print(f"Aggregated shape: {df_agg.shape}")
print("Groups per model:")
print(df_agg.groupby("model").size().to_string())
df_agg.head()

In [ ]:
# Summary table: for each (model, temperature), bootstrap CIs for key metrics
rng_summary = np.random.default_rng(SEED)
summary_rows = []

for model_name in df_agg["model"].unique():
    for temp in sorted(df_agg[df_agg["model"] == model_name]["temperature"].unique()):
        mask = (df_agg["model"] == model_name) & (df_agg["temperature"] == temp)
        sub = df_agg[mask]
        row = {"model": model_name, "temperature": temp, "n_groups": len(sub)}

        for metric_col in ["ndcg@10_mean", "hit_rate@10_mean", "ndcg@10_cv"]:
            vals = sub[metric_col].dropna().to_numpy()
            mean, ci_low, ci_high = bootstrap_mean_ci(vals, rng_summary)
            row[f"{metric_col}"] = mean
            row[f"{metric_col}_ci_low"] = ci_low
            row[f"{metric_col}_ci_high"] = ci_high

        for div_col in ["normalized_expected_distance", "cosine_diversity"]:
            vals = sub[div_col].dropna().to_numpy()
            mean, ci_low, ci_high = bootstrap_mean_ci(vals, rng_summary)
            row[div_col] = mean
            row[f"{div_col}_ci_low"] = ci_low
            row[f"{div_col}_ci_high"] = ci_high

        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("Summary table:")
display_cols = [
    "model",
    "temperature",
    "n_groups",
    "ndcg@10_mean",
    "ndcg@10_mean_ci_low",
    "ndcg@10_mean_ci_high",
    "hit_rate@10_mean",
    "ndcg@10_cv",
    "normalized_expected_distance",
]
print(df_summary[display_cols].to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Determinism check at temp=0
# Verify std of quality metrics ~ 0 at temp=0 per model
df_temp0 = df_agg[df_agg["temperature"] == 0.0]

if not df_temp0.empty:
    print("Determinism check at temperature=0.0:")
    print("(std should be ~0 for HF, near-0 for API)\n")

    for model_name in sorted(df_temp0["model"].unique()):
        sub = df_temp0[df_temp0["model"] == model_name]
        ndcg_std_mean = sub["ndcg@10_std"].mean()
        hr_std_mean = sub["hit_rate@10_std"].mean()
        flag = " *** NON-DETERMINISTIC" if ndcg_std_mean > 0.01 else ""
        print(
            f"  {model_name:25s} | avg ndcg@10 std: {ndcg_std_mean:.6f} "
            f"| avg hr@10 std: {hr_std_mean:.6f}{flag}"
        )
else:
    print("No temp=0.0 data found.")

In [ ]:
# Deviation from uniform: t-test + Wilcoxon on normalized_expected_distance vs 1.0
def analyze_deviation_from_uniform(
    df_metrics: pd.DataFrame,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """Statistical analysis of deviation from uniform baseline per (model, temperature)."""
    results: list[dict] = []
    for (model_name, t), sub in df_metrics.groupby(["model", "temperature"]):
        e_norm = sub["normalized_expected_distance"].dropna()
        if len(e_norm) > 1:
            t_stat, p_ttest = ttest_1samp(e_norm, 1.0)
            _, p_wilcoxon = wilcoxon(e_norm - 1.0)
            effect_size = (e_norm.mean() - 1.0) / e_norm.std()
            results.append(
                {
                    "model": model_name,
                    "temperature": t,
                    "n": len(e_norm),
                    "e_norm_mean": e_norm.mean(),
                    "e_norm_std": e_norm.std(),
                    "t_statistic": t_stat,
                    "p_ttest": p_ttest,
                    "p_wilcoxon": p_wilcoxon,
                    "cohens_d": effect_size,
                    "sig_ttest": p_ttest < alpha,
                    "sig_wilcoxon": p_wilcoxon < alpha,
                    "direction": "concentrated" if e_norm.mean() < 1.0 else "diverse",
                }
            )
    return pd.DataFrame(results)


df_deviation = analyze_deviation_from_uniform(df_agg)
print("Deviation from uniform baseline:")
print(
    df_deviation[
        [
            "model",
            "temperature",
            "e_norm_mean",
            "e_norm_std",
            "p_ttest",
            "cohens_d",
            "sig_ttest",
            "direction",
        ]
    ].to_string(index=False, float_format="{:.4f}".format)
)

In [ ]:
# Bootstrap CIs for diversity (normalized_expected_distance) per (model, temperature)
rng_boot = np.random.default_rng(SEED)
boot_rows = []

for (model_name, temp), sub in df_agg.groupby(["model", "temperature"]):
    e_norm = sub["normalized_expected_distance"].dropna().to_numpy()
    if len(e_norm) > 1:
        mean_val, ci_low, ci_high = bootstrap_mean_ci(e_norm, rng_boot)
        contains_uniform = ci_low <= 1.0 <= ci_high
        boot_rows.append(
            {
                "model": model_name,
                "temperature": temp,
                "mean_e_norm": mean_val,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "contains_uniform": contains_uniform,
                "sig_different": not contains_uniform,
            }
        )

df_boot = pd.DataFrame(boot_rows)
print("Bootstrap CIs for normalized expected distance:")
print(df_boot.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Stability ranking: rank models by average CV of ndcg@10 (lower = more stable)
# Exclude temp=0 (deterministic) for ranking
df_nonzero = df_agg[df_agg["temperature"] > 0.0]

if not df_nonzero.empty:
    ranking = (
        df_nonzero.groupby("model")["ndcg@10_cv"].mean().sort_values().reset_index()
    )
    ranking.columns = ["model", "avg_cv_ndcg10"]
    ranking["rank"] = range(1, len(ranking) + 1)
    print("Stability ranking (lower CV = more stable):")
    print(ranking.to_string(index=False, float_format="{:.4f}".format))
else:
    print("No non-zero temperature data for ranking.")

In [ ]:
# Save aggregated results
agg_path = OUTPUT_PATH / f"stability_aggregated_{now}.parquet"
df_agg.to_parquet(agg_path, index=False)
print(f"Aggregated results saved to {agg_path}")
print(f"Shape: {df_agg.shape}")

## Visualizations

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
(OUTPUT_PATH / "figures").mkdir(parents=True, exist_ok=True)

In [ ]:
# Quality vs temperature per model: line plot with bootstrap CIs for ndcg@10
fig, ax = plt.subplots(figsize=(10, 6))
rng_viz = np.random.default_rng(SEED)

for model_name in sorted(df_agg["model"].unique()):
    sub = df_agg[df_agg["model"] == model_name]
    temps_sorted = sorted(sub["temperature"].unique())
    means, lows, highs = [], [], []
    for t in temps_sorted:
        vals = sub[sub["temperature"] == t]["ndcg@10_mean"].dropna().to_numpy()
        m, lo, hi = bootstrap_mean_ci(vals, rng_viz)
        means.append(m)
        lows.append(lo)
        highs.append(hi)
    ax.plot(temps_sorted, means, marker="o", label=model_name)
    ax.fill_between(temps_sorted, lows, highs, alpha=0.1)

ax.set_xlabel("Temperature")
ax.set_ylabel("NDCG@10 (mean across prompts)")
ax.set_title("Quality vs Temperature per Model")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig(
    OUTPUT_PATH / "figures" / "quality_vs_temp.png", dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# Stability vs temperature per model: CV of ndcg@10
fig, ax = plt.subplots(figsize=(10, 6))

for model_name in sorted(df_agg["model"].unique()):
    sub = df_agg[df_agg["model"] == model_name]
    temps_sorted = sorted(sub["temperature"].unique())
    cv_means = []
    for t in temps_sorted:
        vals = sub[sub["temperature"] == t]["ndcg@10_cv"].dropna().to_numpy()
        cv_means.append(float(np.nanmean(vals)) if len(vals) > 0 else np.nan)
    ax.plot(temps_sorted, cv_means, marker="s", label=model_name)

ax.set_xlabel("Temperature")
ax.set_ylabel("CV of NDCG@10 (lower = more stable)")
ax.set_title("Stability vs Temperature per Model")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig(
    OUTPUT_PATH / "figures" / "stability_vs_temp.png", dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# Mean-variance scatter: ndcg@10 mean vs std, colored by model
fig, ax = plt.subplots(figsize=(10, 7))

df_nonzero_viz = df_agg[df_agg["temperature"] > 0.0].copy()
if not df_nonzero_viz.empty:
    models = sorted(df_nonzero_viz["model"].unique())
    palette = sns.color_palette("tab10", n_colors=len(models))

    for i, model_name in enumerate(models):
        sub = df_nonzero_viz[df_nonzero_viz["model"] == model_name]
        ax.scatter(
            sub["ndcg@10_mean"],
            sub["ndcg@10_std"],
            label=model_name,
            alpha=0.6,
            s=60,
            color=palette[i],
            edgecolors="black",
            linewidths=0.3,
        )

    ax.set_xlabel("NDCG@10 (mean)")
    ax.set_ylabel("NDCG@10 (std)")
    ax.set_title("Mean-Variance Scatter (temp > 0)")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
    ax.text(
        0.95,
        0.05,
        "IDEAL:\nHigh mean\nLow std",
        transform=ax.transAxes,
        fontsize=10,
        va="bottom",
        ha="right",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.5},
    )

plt.tight_layout()
fig.savefig(
    OUTPUT_PATH / "figures" / "mean_variance_scatter.png", dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# Diversity heatmap: normalized_expected_distance as model x temperature
pivot = df_agg.pivot_table(
    index="model",
    columns="temperature",
    values="normalized_expected_distance",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    center=1.0,
    ax=ax,
    cbar_kws={"label": "Normalized Expected Distance"},
)
ax.set_title("Diversity: Normalized Expected Distance\n(1.0 = uniform baseline)")
ax.set_ylabel("Model")
ax.set_xlabel("Temperature")
plt.tight_layout()
fig.savefig(
    OUTPUT_PATH / "figures" / "diversity_heatmap.png", dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# Correlation heatmap: Spearman correlation of diversity + entropy metrics
corr_cols = [
    "gini",
    "label_entropy",
    "variation_ratio",
    "normalized_expected_distance",
    "mean_pairwise_cosine_distance",
    "cosine_diversity",
    "normalized_entropy_mean",
    "unique_token_ratio_mean",
    "ndcg@10_cv",
]
available_cols = [c for c in corr_cols if c in df_agg.columns]
corr = df_agg[available_cols].corr(method="spearman", min_periods=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    cmap="RdBu_r",
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_title("Spearman Correlation: Diversity + Entropy Metrics")
plt.tight_layout()
fig.savefig(
    OUTPUT_PATH / "figures" / "correlation_heatmap.png", dpi=150, bbox_inches="tight"
)
plt.show()